<div align="right"><sub>Notebook 最終更新: 2026-03-25 16:33</sub></div>
<h1><strong>05. AIエージェントの基礎</strong></h1>

今回からは，LLMを単体で使うのではなく，役割を持った「エージェント」として組み合わせて，複雑なタスクをこなす方法を学びます．
まずは，回答を行う **Executor** と，その回答をチェックする **Critic** の2役を組み合わせた「自己修正ループ」を体験しましょう．

> **モデルの変更**: エージェントには高い推論・メタ認知能力が求められるため，この回から **Qwen3-8B-Instruct** を使用します（01〜04回は Qwen2.5-3B-Instruct）．

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentence-transformers faiss-cpu peft datasets gradio

import os
import sys
from google.colab import drive

DRIVE_MOUNT_POINT = '/content/drive'
drive.mount(DRIVE_MOUNT_POINT, force_remount=False)

PERSIST_ROOT = os.path.join(DRIVE_MOUNT_POINT, 'MyDrive', 'AIAgent')
PERSIST_INDEX_DIR = os.path.join(PERSIST_ROOT, 'data', 'index')
os.makedirs(PERSIST_INDEX_DIR, exist_ok=True)

REPO_ROOT = '/content/llm_lab'
if not os.path.exists(REPO_ROOT):
    !git clone -b ai_agent https://github.com/akio-kobayashi/llm_lab.git {REPO_ROOT}

os.chdir(REPO_ROOT)
src_path = os.path.abspath('src')
if src_path not in sys.path:
    sys.path.append(src_path)

print('現在の作業ディレクトリ:', os.getcwd())
print('永続ディレクトリ:', PERSIST_ROOT)
from src.common import load_llm, generate_text, AGENT_MODEL_ID
from src.agent_core import LLMExecutorCriticAgent, RoleConfig

model, tokenizer = load_llm(model_id=AGENT_MODEL_ID)
print('準備完了')


## **1. チャット関数の準備**
日本語LLMのチャット機能を使って，システムプロンプトを受け取れるラッパー関数を作成します．

In [ ]:
def llm_chat(system_prompt: str, user_prompt: str, max_tokens: int = 512, temp: float = 0.3):
    return generate_text(model, tokenizer, user_prompt, max_new_tokens=max_tokens, temperature=temp, system_prompt=system_prompt)

## **2. 2役エージェントの実行**
難しい論理問題やプログラミングの質問を投げて，Critic がどのように間違いを見つけ，Executor がそれを修正するか観察します．

In [ ]:
agent = LLMExecutorCriticAgent(llm_chat)

user_query = "次の案内文を、(1) 日時、(2) 場所、(3) 参加条件、(4) です・ます調、(5) 80字以内、の5条件を満たすように書き直してください。案内文: 研究室見学をする予定です。来たい人は参加できますが、申込みした人を優先します。今週土曜日の午後2時からで、場所は情報学部1号館3階の301室です。"#@param{type:'string'}

final_answer, full_log, steps = agent.run_pipeline(user_query)

print("=== エージェントの処理過程 ===")
print(full_log)

print("\n=== 最終回答 ===")
print(final_answer)
print(f"\n文字数: {len(final_answer)}字")

### ✅ 観察ポイント
- Executor は5条件をすべて満たした回答を出力できていますか？
- Critic は問題があれば的確に指摘し，問題なければ「誤りなし」と回答していますか？
- Critic が指摘した場合，修正後の回答は改善されていますか？

## **3. エージェントの効果**
Critic のプロンプトを，条件漏れや曖昧さに厳しい校閲者向けに変更し，回答の質がどう変わるか観察してみましょう．

In [ ]:
executor_system_prompt = "あなたは丁寧な文章作成者です。必ず日本語で回答してください。元の情報を保ちながら、読みやすく簡潔に書き直してください。回答は書き直した文章のみを出力し、複数のバージョンや解説は不要です。"#@param{type:'string'}
critic_system_prompt = "あなたは文章の校閲者です。必ず日本語で回答してください。以下の5項目を1つずつチェックし、各項目に ✓（満たしている）または ✗（満たしていない）を付けて報告してください。\n(1) 日時が含まれているか\n(2) 場所が含まれているか\n(3) 参加条件が含まれているか\n(4) です・ます調になっているか\n(5) 80字以内か（文字数を数えて報告すること）\nすべて ✓ なら「誤りなし」と最後に追記してください。✗ がある場合は最小限の修正案を提示してください。元の文に書かれていない情報を「不足」と指摘しないでください。"#@param{type:'string'}
custom_critic = RoleConfig(
    name="Writing Critic", 
    system_prompt=critic_system_prompt
)
custom_executor = RoleConfig(name="Executor", system_prompt=executor_system_prompt)
agent_custom = LLMExecutorCriticAgent(llm_chat, role_configs=[custom_executor, custom_critic])

answer, log, _ = agent_custom.run_pipeline(user_query, max_iterations=2)
print(log)
print("\n=== 最終回答 ===")
print(answer)
print(f"\n文字数: {len(answer)}字")

### 📊 比較
セクション2（デフォルトプロンプト）とセクション3（チェックリスト付き Critic）の結果を比較してみましょう：

| 観点 | デフォルト | チェックリスト版 |
|:--|:--|:--|
| Executor の出力形式 | | |
| Critic の指摘の正確さ | | |
| 文字数制限の遵守 | | |
| 最終回答の5条件充足 | | |

## **まとめ**
- 1つのプロンプトで完璧な回答を求める（Zero-shot）よりも，役割を分けて「自分で自分のミスを直す」プロセスを入れることで，より信頼性の高い回答が得られるようになります．
- **Critic の役割** は「他の回答を客観的に評価する」というメタ認知的なタスクであり，Executor より高い推論能力が求められます．そのため，エージェントパターンには一定以上のモデル規模が必要です．
- **プロンプトの設計** も重要です．チェックリスト形式のプロンプトを Critic に与えることで，的確で構造化された評価が可能になります．